# NHANES: contexto, diccionario de variables y preparación para ML

Este cuadernillo trabaja con el archivo `NHANES_Select_Mean_Dietary_Intake_Estimates_20260407.csv`.

## Contexto general

NHANES (National Health and Nutrition Examination Survey) es una encuesta nacional de EE. UU. que monitorea salud y nutrición. Este dataset contiene **estimaciones promedio de ingesta dietaria** para distintos grupos poblacionales (sexo, edad, raza/origen hispano), en varios periodos de encuesta.

Cada fila representa una combinación de:
- periodo de encuesta,
- grupo demográfico,
- nutriente,
- y su estimación media con incertidumbre estadística (error estándar e intervalo de confianza 95%).

Como objetivo de ciencia de datos, prepararemos una versión numérica y limpia para utilizar en:
- **Regresión** (predecir un valor continuo como `Mean`),
- **Clasificación** (predecir una clase derivada, por ejemplo nivel de ingesta),
- **Redes neuronales** (entrada totalmente numérica y escalada).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

DATA_PATH = Path('NHANES_Select_Mean_Dietary_Intake_Estimates_20260407.csv')
df_raw = pd.read_csv(DATA_PATH)

print('Shape:', df_raw.shape)
display(df_raw.head())

## Tabla de columnas: significado del dataset

Primero construimos una tabla descriptiva para documentar cada columna.

In [ ]:
column_dictionary = pd.DataFrame([
    ['Survey Years', 'Categorica (texto)', 'Periodo NHANES de 2 anos (ej. 1999-2000, 2017-2018).'],
    ['Sex', 'Categorica (texto)', 'Sexo del grupo: All, Female o Male.'],
    ['Age Group', 'Categorica (texto)', 'Rango etario del grupo (ej. 2-5, 20 and over, 70 and over).'],
    ['Race and Hispanic Origin', 'Categorica (texto)', 'Subgrupo racial/etnico (All, Non-Hispanic White, etc.).'],
    ['Nutrient', 'Categorica (texto)', 'Nutriente evaluado (Calcium, Iron, Sodium, etc.).'],
    ['Mean', 'Numerica', 'Estimacion media de ingesta del nutriente para el grupo.'],
    ['Standard Error', 'Numerica', 'Error estandar de la estimacion media.'],
    ['Lower 95% CI Limit', 'Numerica', 'Limite inferior del intervalo de confianza al 95%.'],
    ['Upper 95% CI Limit', 'Numerica', 'Limite superior del intervalo de confianza al 95%.'],
    ['Note1', 'Texto / nota metodologica', 'Notas de disponibilidad o advertencias para la fila.'],
    ['Note2', 'Texto / nota metodologica', 'Segunda nota metodologica (cuando aplica).'],
    ['Notea', 'Texto simbolico', 'Marca simbolica de nota (ej. *).'],
    ['Noteb', 'Texto simbolico', 'Marca simbolica adicional (ej. dagger).'],
], columns=['Columna', 'Tipo original', 'Significado'])

display(column_dictionary)

## Exploracion inicial

Revisamos tipos, nulos y cantidad de categorias para entender calidad y estructura.

In [ ]:
display(df_raw.info())

nulls = df_raw.isna().sum().sort_values(ascending=False)
display(pd.DataFrame({'nulos': nulls, 'pct_nulos': (nulls / len(df_raw)).round(4)}))

cat_cols = ['Survey Years', 'Sex', 'Age Group', 'Race and Hispanic Origin', 'Nutrient']
for c in cat_cols:
    print(f'\n{c}: {df_raw[c].nunique()} categorias')
    print(df_raw[c].dropna().unique()[:15])

## Limpieza y transformacion a formato numerico

### Que se transforma y por que

1. **Columnas numericas con comas** (`Mean`, `Standard Error`, limites CI):
   - vienen como texto en algunas filas por formato `1,051.9`.
   - se eliminan comas y se convierten a `float` para poder modelar.

2. **`Survey Years`**:
   - se descompone en `survey_start_year`, `survey_end_year`, `survey_mid_year`.
   - esto da una representacion temporal numerica para modelos.

3. **`Age Group`**:
   - se extraen `age_min`, `age_max`, `age_mid`.
   - para grupos abiertos (`20 and over`, `70 and over`) se asigna un maximo operativo (`90`) para permitir representacion numerica estable.

4. **Notas (`Note1`, `Note2`, `Notea`, `Noteb`)**:
   - contienen metadatos de disponibilidad y banderas.
   - se transforman en indicadores binarios (`has_note1`, etc.) y adicionalmente en una bandera `is_data_missing_by_design`.

5. **Categoricas principales** (`Sex`, `Race and Hispanic Origin`, `Nutrient`):
   - se mantienen para interpretabilidad,
   - y se codifican con **one-hot encoding** para modelos que requieren entrada numerica.

In [ ]:
df = df_raw.copy()

# 1) Normalizar columnas numericas con comas
num_cols = ['Mean', 'Standard Error', 'Lower 95% CI Limit', 'Upper 95% CI Limit']
for col in num_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .replace({'nan': np.nan, 'None': np.nan, '': np.nan})
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 2) Survey years a variables numericas
years = df['Survey Years'].str.extract(r'(?P<start>\d{4})-(?P<end>\d{4})')
df['survey_start_year'] = pd.to_numeric(years['start'], errors='coerce')
df['survey_end_year'] = pd.to_numeric(years['end'], errors='coerce')
df['survey_mid_year'] = (df['survey_start_year'] + df['survey_end_year']) / 2

# 3) Parseo de edad
def parse_age_group(text):
    if pd.isna(text):
        return np.nan, np.nan
    s = str(text).strip().lower()
    if 'and over' in s:
        age_min = pd.to_numeric(''.join(ch for ch in s if ch.isdigit()), errors='coerce')
        return age_min, 90.0
    if '-' in s:
        a, b = s.split('-', 1)
        return pd.to_numeric(a, errors='coerce'), pd.to_numeric(b, errors='coerce')
    return np.nan, np.nan

age_parsed = df['Age Group'].apply(parse_age_group)
df['age_min'] = age_parsed.apply(lambda x: x[0])
df['age_max'] = age_parsed.apply(lambda x: x[1])
df['age_mid'] = (df['age_min'] + df['age_max']) / 2

# 4) Indicadores de notas
for note_col in ['Note1', 'Note2', 'Notea', 'Noteb']:
    df[f'has_{note_col.lower()}'] = df[note_col].notna().astype(int)

df['is_data_missing_by_design'] = (
    df[['Note1', 'Note2']]
    .fillna('')
    .agg(' '.join, axis=1)
    .str.contains('only available|only available from', case=False, regex=True)
    .astype(int)
)

display(df.head())
display(df[num_cols + ['survey_mid_year', 'age_min', 'age_max', 'age_mid']].describe().T)

## Construccion de dataset numerico para ML

Creamos `df_ml` con variables codificadas para algoritmos supervisados.

- Se eliminan columnas de texto libre de notas para evitar ruido semantico no estructurado.
- Se aplica **one-hot encoding** a categoricas.
- Se conserva `Mean` como target base para regresion.
- Se crea target de clasificacion `mean_level` por terciles.

In [ ]:
df_model_base = df.copy()

# Filas sin target numerico para regresion
df_model_base = df_model_base[df_model_base['Mean'].notna()].copy()

# Clasificacion: discretizar Mean en 3 niveles
df_model_base['mean_level'] = pd.qcut(
    df_model_base['Mean'],
    q=3,
    labels=['bajo', 'medio', 'alto']
)

categorical_for_encoding = ['Sex', 'Age Group', 'Race and Hispanic Origin', 'Nutrient', 'Survey Years']
numeric_keep = [
    'Mean', 'Standard Error', 'Lower 95% CI Limit', 'Upper 95% CI Limit',
    'survey_start_year', 'survey_end_year', 'survey_mid_year',
    'age_min', 'age_max', 'age_mid',
    'has_note1', 'has_note2', 'has_notea', 'has_noteb', 'is_data_missing_by_design'
]

df_ml = pd.get_dummies(
    df_model_base[numeric_keep + categorical_for_encoding + ['mean_level']],
    columns=categorical_for_encoding,
    drop_first=False
)

print('Shape df_ml:', df_ml.shape)
display(df_ml.head())

## Ejemplo de preparacion final para 3 tipos de modelos

Separamos en conjuntos para:
- Regresion (`y_reg = Mean`)
- Clasificacion (`y_clf = mean_level_*`)
- Redes neuronales (mismos datos numericamente escalados)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Regresion ---
reg_drop = ['Mean'] + [c for c in df_ml.columns if c.startswith('mean_level_')]
X_reg = df_ml.drop(columns=reg_drop)
y_reg = df_ml['Mean']

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# --- Clasificacion ---
target_clf_col = [c for c in df_ml.columns if c.startswith('mean_level_')]
if len(target_clf_col) == 0:
    raise ValueError('No se encontro variable de clasificacion en one-hot.')

# Reconstruir etiqueta de clase desde one-hot
y_clf = df_ml[target_clf_col].idxmax(axis=1).str.replace('mean_level_', '', regex=False)
X_clf = df_ml.drop(columns=['Mean'] + target_clf_col)

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# --- Escalado util para redes neuronales (y muchos modelos) ---
scaler_reg = StandardScaler()
Xr_train_scaled = scaler_reg.fit_transform(Xr_train)
Xr_test_scaled = scaler_reg.transform(Xr_test)

scaler_clf = StandardScaler()
Xc_train_scaled = scaler_clf.fit_transform(Xc_train)
Xc_test_scaled = scaler_clf.transform(Xc_test)

print('Regresion -> X_train:', Xr_train.shape, '| y_train:', yr_train.shape)
print('Clasificacion -> X_train:', Xc_train.shape, '| y_train:', yc_train.shape)
print('Matrices escaladas para NN (reg):', Xr_train_scaled.shape)
print('Matrices escaladas para NN (clf):', Xc_train_scaled.shape)

## Resumen de cambios aplicados al dataset

- **Texto numerico a float**: se removieron comas de miles y se convirtio a numerico en `Mean`, `Standard Error`, `Lower 95% CI Limit`, `Upper 95% CI Limit`.
- **Tiempo a variables modelables**: `Survey Years` se separo en inicio, fin y punto medio.
- **Edad a escala cuantitativa**: `Age Group` se tradujo a minimo, maximo y media etaria aproximada.
- **Notas a banderas binarias**: se crearon `has_note*` e `is_data_missing_by_design` para preservar informacion metodologica.
- **Categorias a numerico (one-hot)**: `Sex`, `Age Group`, `Race and Hispanic Origin`, `Nutrient` y `Survey Years` se codificaron a columnas binarias.
- **Targets listos**: `Mean` para regresion y `mean_level` para clasificacion.

Con esto ya tienes una base consistente para entrenar modelos clasicos y redes neuronales.